In [16]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

path = path = "C:/Users/moham/olist-project/data/clean/"

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]

orders  = pd.read_csv(path + "orders_delivered.csv", parse_dates=date_cols)
reviews = pd.read_csv(path + "reviews_clean.csv")
items   = pd.read_csv(path + "order_items.csv")

print(len(orders), len(reviews), len(items))

96470 98673 112650


In [20]:
count = np.array([36, 913])        # عملاء رجعوا: متأخر، في الميعاد
nobs  = np.array([2654, 49644])    # إجمالي العملاء في كل مجموعة

p_late, p_ontime = count / nobs
print(f"Late: {p_late:.2%} | On-time: {p_ontime:.2%}")


z, p_z = proportions_ztest(count, nobs)
print(f"z = {z:.2f}, p = {p_z:.3f}")


table = np.array([[36, 2654 - 36],
                  [913, 49644 - 913]])
chi2, p_chi, dof, expected = stats.chi2_contingency(table, correction=False)
chi2_y, p_chi_y, _, _ = stats.chi2_contingency(table)  
print(f"chi2 = {chi2:.2f}, p = {p_chi:.3f} | with Yates: p = {p_chi_y:.3f}")


diff = p_late - p_ontime
se = np.sqrt(p_late * (1 - p_late) / nobs[0] + p_ontime * (1 - p_ontime) / nobs[1])
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se
print(f"Diff = {diff*100:.2f} pp, 95% CI = [{ci_low*100:.2f}, {ci_high*100:.2f}] pp")

print("Expected returners if same rate:", round(nobs[0] * p_ontime, 1), "| actual:", count[0])

Late: 1.36% | On-time: 1.84%
z = -1.81, p = 0.070
chi2 = 3.29, p = 0.070 | with Yates: p = 0.082
Diff = -0.48 pp, 95% CI = [-0.94, -0.03] pp
Expected returners if same rate: 48.8 | actual: 36


In [21]:
df = orders.merge(reviews[["order_id", "review_score"]], on="order_id", how="inner")
print(len(df)) 

df["late"] = df["delay_days"] > 0
df["negative"] = df["review_score"] <= 2

summary = df.groupby("late").agg(
    orders=("order_id", "count"),
    mean_score=("review_score", "mean"),
    median_score=("review_score", "median"),
    negative_pct=("negative", lambda s: s.mean() * 100),
).round(2)
print(summary)

late_scores   = df.loc[df["late"], "review_score"]
ontime_scores = df.loc[~df["late"], "review_score"]

u, p_mw = stats.mannwhitneyu(late_scores, ontime_scores, alternative="two-sided")
cles = u / (len(late_scores) * len(ontime_scores))
mean_diff = late_scores.mean() - ontime_scores.mean()

print(f"Mann-Whitney p = {p_mw:.3g}")
print(f"Mean difference = {mean_diff:.2f} stars")
print(f"P(late score > on-time score, ties = half) = {cles:.2f}")

95824
       orders  mean_score  median_score  negative_pct
late                                                 
False   89443        4.29           5.0          9.27
True     6381        2.27           1.0         62.42
Mann-Whitney p = 0
Mean difference = -2.02 stars
P(late score > on-time score, ties = half) = 0.18


In [22]:
order_revenue = (items.groupby("order_id", as_index=False)["price"].sum()
                      .rename(columns={"price": "revenue"}))


check = orders[["order_id"]].merge(order_revenue, on="order_id", how="left")
print(round(check["revenue"].sum(), 2), "| missing:", check["revenue"].isna().sum())

df = df.merge(order_revenue, on="order_id", how="left")


baseline  = df.loc[~df["late"], "negative"].mean()
late_rate = df.loc[df["late"], "negative"].mean()
attributable = (late_rate - baseline) / late_rate

rev_late_neg = df.loc[df["late"] & df["negative"], "revenue"].sum()
rev_at_risk  = rev_late_neg * attributable
total_rev    = df["revenue"].sum()

print(f"Baseline negative rate: {baseline:.1%} | Late negative rate: {late_rate:.1%}")
print(f"Share of negative reviews attributable to delay: {attributable:.1%}")
print(f"Revenue of late+negative orders: {rev_late_neg:,.0f}")
print(f"Excess revenue at risk: {rev_at_risk:,.0f} ({rev_at_risk/total_rev:.1%} of reviewed revenue)")

13220248.93 | missing: 0
Baseline negative rate: 9.3% | Late negative rate: 62.4%
Share of negative reviews attributable to delay: 85.2%
Revenue of late+negative orders: 619,012
Excess revenue at risk: 527,109 (4.0% of reviewed revenue)
